# Milestone 3 · TV5 — Publication Split & Leakage Guard (Colab CPU)

Notebook chạy trên **Google Colab CPU** (không cần GPU, không cần cài thư viện ngoài) để thực hiện bước chia tập **`train / valid / test` theo publication** của **TV5** ngay sau khi **TV3** bàn giao **Auto Pool (`T3-05`)**.

### Quy tắc áp dụng theo `docs/MS3_CONTRACTS.md` (nhóm `k = 5`)
- **Group strictly by `arxiv_id`**: 1 publication chỉ thuộc đúng 1 partition (`train`, `valid`, hoặc `test`).
- **Disjoint pools**: tập `manual` (T3-04) và tập `heuristic` / `auto` (T3-05) không giao nhau.
- **Quota chuẩn:**
  - `manual` pool: `≥ 25` bài, mỗi bài `≥ 20` BibTeX entries `accepted`.
  - `auto` pool: `≥ max(ceil(0.1 × (N − M)), 10)` bài (với `N = 16,321`, `M = 25` → tối thiểu `1,630` bài).
  - `valid`: **5 manual + 5 auto** (10 bài).
  - `test`: **5 manual + 5 auto** (10 bài).
  - `train`: toàn bộ các bài còn lại.
- **Chống rò rỉ nhãn `test`:** Notebook tự động xuất riêng gói `handoff_tv2_tv4.zip` (**chỉ chứa `partition_manifest.json` và `labels_train_valid.json`**, **không chứa nhãn `test`**) để bàn giao cho TV2/TV4; còn `labels_test_tv5_only.json` do TV5 giữ kín đến bước đánh giá cuối.

> Chạy lần lượt **Cell 1 → Cell 5** từ trên xuống. Dùng đường dẫn Drive đã mount (`/content/drive/MyDrive/...`), **không nhập URL Drive**.


In [ ]:
# @title 1. Mount Google Drive và tải code arxiv-data-science
MOUNT_DRIVE = True  # @param {type:"boolean"}
REPO_URL = "https://github.com/LeoTkiet/arxiv-data-science.git"  # @param {type:"string"}
# Commit trên nhánh main đã merge PR #32 (TV5 split/metrics/prediction/evaluation/export)
CODE_REF = "aa581d5113fac872c8fa68608eea26f60ca74e42"  # @param {type:"string"}
REPO_DIR = "/content/arxiv-data-science"

import importlib
import subprocess
import sys
from pathlib import Path

if sys.version_info < (3, 10):
    raise RuntimeError("Cần Python 3.10 trở lên.")

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

repo = Path(REPO_DIR)
if not repo.exists():
    subprocess.run(["git", "clone", REPO_URL, str(repo)], check=True)

if CODE_REF.strip():
    subprocess.run(["git", "-C", str(repo), "fetch", "origin", CODE_REF.strip()], check=True)
    subprocess.run(["git", "-C", str(repo), "checkout", "--detach", "FETCH_HEAD"], check=True)

CODE_SHA = subprocess.check_output(
    ["git", "-C", str(repo), "rev-parse", "HEAD"], text=True
).strip()

src_dir = str((repo / "src").resolve())
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)
for mod in list(sys.modules):
    if mod == "arxiv_pipeline" or mod.startswith("arxiv_pipeline."):
        sys.modules.pop(mod, None)
importlib.invalidate_caches()

from arxiv_pipeline.matching.split import (
    assert_no_split_leakage,
    create_split,
    load_partition_manifest,
    save_partition_manifest,
)
from arxiv_pipeline.matching.contracts import ContractError, LabelRecord, canonical_id

print("Code commit:", CODE_SHA)
print("Import arxiv_pipeline.matching.split: OK")


## 2. Cấu hình đường dẫn file đầu vào & thư mục lưu kết quả

- `POPULATION_MANIFEST_PATH`: file `population_manifest.json` của nhóm (hỗ trợ cả key `population_ids` của TV3 lẫn `publication_ids` theo contract `3.0`).
- `MANUAL_LABELS_PATH`: trỏ tới `labels.json` (hoặc `handoff.zip`) trong `04_KET_QUA_COLAB/20261124T094434Z-7edcac30`. Nếu TV3 đã gộp sẵn cả manual + auto vào 1 file duy nhất thì trỏ file đó vào đây và **để trống `AUTO_LABELS_PATH`**.
- `AUTO_LABELS_PATH`: file `auto_labels.json` / `.jsonl` của T3-05 từ TV3 (để trống nếu đã gộp chung ở `MANUAL_LABELS_PATH`).
- `RUN_ID`: để trống sẽ tự sinh theo giờ UTC (`YYYYMMDDTHHMMSSZ-<hex>`), không bao giờ ghi đè run cũ.


In [ ]:
# @title 2. Cấu hình đường dẫn Drive & tham số Split
POPULATION_MANIFEST_PATH = "/content/drive/MyDrive/NMKHDL_MS3_T3-05_Auto_Pool/population_manifest.json"  # @param {type:"string"}
MANUAL_LABELS_PATH = "/content/drive/MyDrive/NMKHDL_MS3_T3-04_Gan_nhan_2026-11-22/04_KET_QUA_COLAB/20261124T094434Z-7edcac30/labels.json"  # @param {type:"string"}
AUTO_LABELS_PATH = "/content/drive/MyDrive/NMKHDL_MS3_T3-05_Auto_Pool/auto_labels.json"  # @param {type:"string"}
OUTPUT_ROOT = "/content/drive/MyDrive/NMKHDL_MS3_TV5_Split"  # @param {type:"string"}
SEED = 42  # @param {type:"integer"}
GROUP_SIZE = 5  # @param {type:"integer"}
STRATIFY = True  # @param {type:"boolean"}
STRICT_AUTO_ACCEPTED = True  # @param {type:"boolean"}
RUN_ID = ""  # @param {type:"string"}
LOCAL_ROOT = "/content/ms3-tv5-split-runs"

from datetime import datetime, timezone
import re
import uuid
from pathlib import Path

for label_name, val in [
    ("POPULATION_MANIFEST_PATH", POPULATION_MANIFEST_PATH),
    ("MANUAL_LABELS_PATH", MANUAL_LABELS_PATH),
    ("AUTO_LABELS_PATH", AUTO_LABELS_PATH),
    ("OUTPUT_ROOT", OUTPUT_ROOT),
]:
    if val.strip().startswith(("http://", "https://")):
        raise ValueError(f"{label_name} phải là đường dẫn Drive đã mount (/content/drive/...), không phải URL.")

pop_path = Path(POPULATION_MANIFEST_PATH.strip())
manual_path = Path(MANUAL_LABELS_PATH.strip())
auto_path = Path(AUTO_LABELS_PATH.strip()) if AUTO_LABELS_PATH.strip() else None
output_root = Path(OUTPUT_ROOT.strip())
local_root = Path(LOCAL_ROOT)

for name, p in [("population_manifest", pop_path), ("manual_labels", manual_path)]:
    if not p.is_file():
        raise FileNotFoundError(f"Không tìm thấy {name}: {p}")
if auto_path is not None and not auto_path.is_file():
    raise FileNotFoundError(f"Không tìm thấy auto_labels: {auto_path}")

resolved_run_id = RUN_ID.strip() or (
    datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:8]
)
if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_.-]{0,79}", resolved_run_id):
    raise ValueError(f"RUN_ID không hợp lệ: {resolved_run_id}")

print("RUN_ID               :", resolved_run_id)
print("population_manifest  :", pop_path)
print("manual_labels        :", manual_path)
print("auto_labels          :", auto_path if auto_path else "(đã gộp trong manual_labels)")
print("output_root          :", output_root / resolved_run_id)
print(f"seed={SEED} | group_size={GROUP_SIZE} | stratify={STRATIFY} | strict_auto_accepted={STRICT_AUTO_ACCEPTED}")


In [ ]:
# @title 3. Preflight: Nạp, chuẩn hóa & kiểm định dữ liệu đầu vào
from collections import Counter, defaultdict
import hashlib
import json
import math
import zipfile
from pathlib import Path

def _sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as fh:
        for chunk in iter(lambda: fh.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def _load_population(path: Path) -> tuple[list[str], dict]:
    raw = json.loads(path.read_text(encoding="utf-8-sig"))
    if isinstance(raw, list):
        ids = raw
        meta = {"format": "list", "declared_count": len(ids)}
    elif isinstance(raw, dict):
        ids = raw.get("publication_ids")
        if ids is None:
            ids = raw.get("population_ids")
        if not isinstance(ids, list) or not ids:
            raise ContractError("population_manifest phải chứa list 'publication_ids' hoặc 'population_ids'")
        meta = {
            "format": "dict",
            "schema_version": raw.get("schema_version"),
            "scope": raw.get("scope"),
            "declared_count": raw.get("population_count", raw.get("unique_population_count", len(ids))),
            "declared_auto_min": raw.get("auto_minimum_publications"),
        }
    else:
        raise ContractError(f"Định dạng population_manifest không hợp lệ: {type(raw).__name__}")

    canon = [canonical_id(p) for p in ids]
    if len(canon) != len(set(canon)):
        raise ContractError("population_manifest có arxiv_id bị trùng lặp")
    if meta["declared_count"] != len(canon):
        raise ContractError(
            f"Số lượng ID khai báo ({meta['declared_count']}) != thực tế ({len(canon)})"
        )
    return sorted(canon), meta

def _load_label_records(path: Path) -> list[LabelRecord]:
    if path.suffix.lower() == ".zip":
        with zipfile.ZipFile(path, "r") as zf:
            candidates = [n for n in zf.namelist() if n.endswith("labels.json")]
            if not candidates:
                raise ContractError(f"Không tìm thấy labels.json bên trong {path.name}")
            chosen = "collected/labels.json" if "collected/labels.json" in candidates else candidates[0]
            raw_obj = json.loads(zf.read(chosen).decode("utf-8-sig"))
    elif path.suffix.lower() == ".jsonl":
        raw_obj = [
            json.loads(line)
            for line in path.read_text(encoding="utf-8-sig").splitlines()
            if line.strip()
        ]
    else:
        raw_obj = json.loads(path.read_text(encoding="utf-8-sig"))

    if isinstance(raw_obj, dict):
        for key in ("labels", "records", "rows", "entries"):
            if isinstance(raw_obj.get(key), list):
                raw_obj = raw_obj[key]
                break
    if not isinstance(raw_obj, list):
        raise ContractError(f"{path.name}: cần list các LabelRecord (hoặc object có key 'labels')")

    records: list[LabelRecord] = []
    for idx, item in enumerate(raw_obj):
        if not isinstance(item, dict):
            raise ContractError(f"{path.name}: phần tử #{idx} không phải JSON object")
        records.append(LabelRecord(**item))
    return records

pop_ids, pop_meta = _load_population(pop_path)
pop_set = set(pop_ids)
input_hashes = {"population_manifest": {"file": pop_path.name, "sha256": _sha256_file(pop_path)}}

manual_records = _load_label_records(manual_path)
input_hashes["manual_labels"] = {"file": manual_path.name, "sha256": _sha256_file(manual_path), "rows": len(manual_records)}

auto_records = _load_label_records(auto_path) if auto_path is not None else []
if auto_path is not None:
    input_hashes["auto_labels"] = {"file": auto_path.name, "sha256": _sha256_file(auto_path), "rows": len(auto_records)}

all_labels = manual_records + auto_records

# Kiểm tra trùng QueryKey, trộn pool, phạm vi population_ids và quota
seen_keys = set()
dups = []
sources_by_pub = defaultdict(set)
sup_by_pub = Counter()
non_sup_auto = []

for lbl in all_labels:
    if lbl.key in seen_keys:
        dups.append(lbl.key)
    seen_keys.add(lbl.key)
    sources_by_pub[lbl.arxiv_id].add(lbl.label_source)
    if lbl.is_supervised:
        sup_by_pub[lbl.arxiv_id] += 1
    elif lbl.label_source == "heuristic":
        non_sup_auto.append(lbl)

if dups:
    raise ContractError(f"Có {len(dups)} query (arxiv_id, citation_key) bị trùng, ví dụ: {dups[:5]}")

mixed_pubs = sorted(pid for pid, srcs in sources_by_pub.items() if len(srcs) > 1)
if mixed_pubs:
    raise ContractError(f"Có {len(mixed_pubs)} bài bị trộn nhãn manual và heuristic: {mixed_pubs[:10]}")

manual_pubs = sorted(pid for pid, srcs in sources_by_pub.items() if srcs == {"manual"})
auto_pubs = sorted(pid for pid, srcs in sources_by_pub.items() if srcs == {"heuristic"})

outside_pop = sorted((set(manual_pubs) | set(auto_pubs)) - pop_set)
if outside_pop:
    raise ContractError(f"Có {len(outside_pop)} bài nằm ngoài population_manifest: {outside_pop[:10]}")

under_20_manual = [(pid, sup_by_pub[pid]) for pid in manual_pubs if sup_by_pub[pid] < 20]
if under_20_manual:
    raise ContractError(f"Có bài manual chưa đủ 20 nhãn accepted: {under_20_manual[:10]}")

if STRICT_AUTO_ACCEPTED and non_sup_auto:
    sample = [(l.arxiv_id, l.citation_key, l.decision, l.review_status) for l in non_sup_auto[:5]]
    raise ContractError(
        f"Auto pool còn {len(non_sup_auto)} dòng chưa accepted/supervised (abstain/pending/uncertain). "
        f"Ví dụ: {sample}. Cần TV3 xử lý trước khi split (hoặc tắt STRICT_AUTO_ACCEPTED để kiểm thử)."
    )

min_manual = 5 * GROUP_SIZE
min_auto = max(math.ceil((len(pop_ids) - len(manual_pubs)) / 10), 2 * GROUP_SIZE)

print("=== PREFLIGHT SUMMARY ===")
print(f"Population N             : {len(pop_ids):,} bài (SHA-256: {input_hashes['population_manifest']['sha256'][:16]}...)")
print(f"Tổng số LabelRecord      : {len(all_labels):,} dòng (supervised: {sum(1 for l in all_labels if l.is_supervised):,})")
print(f"Manual pool (M)          : {len(manual_pubs):,} bài (yêu cầu >= {min_manual})")
print(f"Auto / Heuristic pool    : {len(auto_pubs):,} bài (yêu cầu >= {min_auto})")
print(f"Auto non-supervised rows : {len(non_sup_auto):,}")
print(f"Overlap manual ∩ auto    : 0 | Outside population: 0")

if len(manual_pubs) < min_manual:
    raise ContractError(f"Thiếu bài Manual: có {len(manual_pubs)}, cần >= {min_manual}")
if len(auto_pubs) < min_auto:
    raise ContractError(f"Thiếu bài Auto: có {len(auto_pubs)}, cần >= {min_auto} (theo N={len(pop_ids)}, M={len(manual_pubs)})")
print("Preflight: PASS — Đủ điều kiện chạy create_split!")


In [ ]:
# @title 4. Chạy create_split, kiểm định leakage & tách nhãn train/valid vs test
import json
from collections import Counter

manifest = create_split(
    all_labels,
    pop_ids,
    seed=SEED,
    group_size=GROUP_SIZE,
    stratify=STRATIFY,
    enforce_quotas=True,
)
leakage_report = assert_no_split_leakage(
    manifest,
    labels=all_labels,
    population_count=len(pop_ids),
)

# Ghi thử vào thư mục local mới và đọc lại để xác minh split_id
local_run_dir = local_root / resolved_run_id
local_run_dir.mkdir(parents=True, exist_ok=False)

local_manifest_path = save_partition_manifest(manifest, local_run_dir / "partition_manifest.json")
reloaded = load_partition_manifest(local_manifest_path)
assert reloaded.split_id == manifest.split_id, "Read-back split_id không khớp!"
assert reloaded.counts == manifest.counts, "Read-back counts không khớp!"

# Tách nhãn theo partition để bảo vệ tập test (TV2/TV4 CHỈ nhận train + valid)
part_by_pub = {row.arxiv_id: row.partition for row in manifest.rows}
labels_train_valid = [lbl for lbl in all_labels if part_by_pub.get(lbl.arxiv_id) in {"train", "valid"}]
labels_test_only = [lbl for lbl in all_labels if part_by_pub.get(lbl.arxiv_id) == "test"]

test_pub_ids = {row.arxiv_id for row in manifest.rows if row.partition == "test"}
assert not any(lbl.arxiv_id in test_pub_ids for lbl in labels_train_valid), "LEAKAGE: nhãn test lọt vào train_valid!"
assert all(lbl.arxiv_id in test_pub_ids for lbl in labels_test_only), "Lỗi tách nhãn test!"
assert len(labels_train_valid) + len(labels_test_only) == len(all_labels), "Mất nhãn khi tách!"

def _dump_json(path: Path, obj) -> None:
    path.write_text(json.dumps(obj, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

_dump_json(local_run_dir / "labels_train_valid.json", [l.to_dict() for l in labels_train_valid])
_dump_json(local_run_dir / "labels_test_tv5_only.json", [l.to_dict() for l in labels_test_only])
_dump_json(local_run_dir / "labels_all_merged.json", [l.to_dict() for l in all_labels])

split_report = {
    "schema_version": "3.0",
    "task": "MS3-TV5-SPLIT",
    "run_id": resolved_run_id,
    "code_commit": CODE_SHA,
    "seed": SEED,
    "group_size": GROUP_SIZE,
    "stratify": STRATIFY,
    "strict_auto_accepted": STRICT_AUTO_ACCEPTED,
    "split_id": manifest.split_id,
    "population_snapshot_id": manifest.population_snapshot_id,
    "label_snapshot_id": manifest.label_snapshot_id,
    "population_count": len(pop_ids),
    "population_meta": pop_meta,
    "counts": manifest.counts,
    "gold_coverage": leakage_report["gold_coverage"],
    "disjoint_verified": leakage_report["disjoint_verified"],
    "label_rows": {
        "total": len(all_labels),
        "train_valid": len(labels_train_valid),
        "test_tv5_only": len(labels_test_only),
        "by_source": dict(Counter(l.label_source for l in all_labels)),
        "by_decision": dict(Counter(l.decision for l in all_labels)),
        "by_review_status": dict(Counter(l.review_status for l in all_labels)),
    },
    "inputs": input_hashes,
}
_dump_json(local_run_dir / "split_report.json", split_report)

print("=== SPLIT COMPLETE & LEAKAGE VERIFIED ===")
print("split_id              :", manifest.split_id)
print("population_snapshot_id:", manifest.population_snapshot_id)
print("label_snapshot_id     :", manifest.label_snapshot_id)
print("\nPhân bổ publications theo partition:")
for part in ("train", "valid", "test"):
    c = manifest.counts[part]
    g = leakage_report["gold_coverage"][part]
    part_total = c["manual"] + c["auto"]
    print(
        f"  {part:5s} | total={part_total:4d} (manual={c['manual']:2d}, auto={c['auto']:4d}) "
        f"| gold_pubs={g['gold_publications']:4d} | matched={g['matched_queries']:6d} | unmatched={g['unmatched_queries']:6d}"
    )
print(f"\nNhãn bàn giao TV2/TV4 (train+valid): {len(labels_train_valid):,} dòng")
print(f"Nhãn TV5 giữ kín (test only)       : {len(labels_test_only):,} dòng")


In [ ]:
# @title 5. Đóng gói & lưu kết quả bàn giao lên Google Drive (không ghi đè run cũ)
import json
import shutil
import zipfile

publish_dir = output_root / resolved_run_id
publish_dir.mkdir(parents=True, exist_ok=False)

# 1. Tạo gói ZIP an toàn cho TV2 & TV4 (TUYỆT ĐỐI không chứa labels_test_tv5_only.json)
handoff_files = [
    "partition_manifest.json",
    "labels_train_valid.json",
    "split_report.json",
]
handoff_manifest = {
    "schema_version": "3.0",
    "task": "MS3-TV5-SPLIT-HANDOFF",
    "run_id": resolved_run_id,
    "split_id": manifest.split_id,
    "code_commit": CODE_SHA,
    "test_labels_excluded": True,
    "files": {name: _sha256_file(local_run_dir / name) for name in handoff_files},
}
_dump_json(local_run_dir / "handoff_manifest.json", handoff_manifest)

handoff_zip = publish_dir / "handoff_tv2_tv4.zip"
with zipfile.ZipFile(handoff_zip, "x", compression=zipfile.ZIP_DEFLATED) as zf:
    for name in handoff_files + ["handoff_manifest.json"]:
        zf.write(local_run_dir / name, arcname=name)

# Kiểm tra chắc chắn ZIP bàn giao không chứa file test
with zipfile.ZipFile(handoff_zip, "r") as zf:
    znames = set(zf.namelist())
    assert "labels_test_tv5_only.json" not in znames and "labels_all_merged.json" not in znames

# 2. Sao chép các file lẻ lên thư mục run trên Drive
for name in [
    "partition_manifest.json",
    "split_report.json",
    "labels_train_valid.json",
    "labels_test_tv5_only.json",
    "labels_all_merged.json",
    "handoff_manifest.json",
]:
    shutil.copy2(local_run_dir / name, publish_dir / name)

# 3. Ghi published_manifest.json cuối cùng kèm SHA-256 mọi file
published_files = {
    p.name: _sha256_file(p)
    for p in sorted(publish_dir.iterdir(), key=lambda x: x.name)
    if p.is_file()
}
published_manifest = {
    "schema_version": "3.0",
    "task": "MS3-TV5-SPLIT",
    "run_id": resolved_run_id,
    "split_id": manifest.split_id,
    "handoff_archive": handoff_zip.name,
    "handoff_archive_sha256": published_files[handoff_zip.name],
    "files": published_files,
}
_dump_json(publish_dir / "published_manifest.json", published_manifest)

print("=== ĐÃ LƯU LÊN GOOGLE DRIVE THÀNH CÔNG ===")
print("Thư mục kết quả     :", publish_dir)
print("split_id            :", manifest.split_id)
print("Gửi cho TV2 & TV4   :", handoff_zip)
print("  (SHA-256)         :", published_manifest["handoff_archive_sha256"])
print("TV5 GIỮ KÍN (test)  :", publish_dir / "labels_test_tv5_only.json")
print("\nDanh sách file & SHA-256:")
for fname, fhash in published_files.items():
    print(f"  {fname:30s} | {fhash[:16]}...")
